# NCCL GPU Overlap Trace

This notebook provides the percentage of overlap for communication and compute kernels. Communication kernels are identified by the 'nccl' prefix.

In [ ]:
import pandas as pd
import plotly.offline as pyo

from IPython.display import display, HTML, Markdown

import nsys_display

pd.options.display.float_format = '{:.1f}'.format

display(HTML("<style>.container { width:95% !important; }</style>"))
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pyo.init_notebook_mode()

## Per-kernel traces

The table displays overlap percentages for each kernel, corresponding to the individual rank selected from the drop-down menu.
All time values are in nanoseconds.

In [ ]:
df = pd.read_parquet('rank_trace.parquet')
nsys_display.display_table_per_rank(df)

## communication compute overall

in this overall part, the compute, communication and overlap duration can be considered as a projection duration.
for example if 2 communication kernel has some overlap, the duration is considered only once in nccl.
and the overlaped duration is the overlaped duration across compute and communication.

In [ ]:
overall_df = pd.read_parquet('grouped_type_merge_df.parquet')
display(overall_df)

## Grouped traces

The table presents overlap percentages for each kernel, grouped by kernel name across all ranks.

In [ ]:
grouped_df = pd.read_parquet('grouped_trace.parquet')
display(grouped_df)

## Grouped Traces V2


in original grouped traces, if 2 kernel with same name has some overlaped duration, it will be counted multi times.
in the new grouped graces, it will not be counted multi times. but if 2 different named kernel has some overlap, the overlaped duration is counted multi times, so as the original grouped trace.

In [ ]:
grouped_name_merge = pd.read_parquet('grouped_name_merge.parquet')
grouped_name_merge.sort_values(by='Duration', ascending=False, inplace=True)
display(grouped_name_merge)

## Communication Compute Matrix

the matrix will display communication-compute kernel matrix, kernels are sorted in duration descending order.   
all communication kernels will be shown, and you can select how many top compute kernels to show.    
first define a formated_display function for better display.


In [ ]:
def formated_display(df):
    col_width = 200

    styles = [
        {
            'selector': 'thead th.col_heading',
            'props': [
                ('position', 'sticky'),
                ('top', '0'),
                ('background-color', 'white'),
                ('z-index', '5'),
                ('white-space', 'normal'),
                ('word-break', 'break-word'),
            ]
        },
        {
            'selector': 'tbody th.row_heading',
            'props': [
                ('position', 'sticky'),
                ('left', '0'),
                ('background-color', 'white'),
                ('z-index', '6'),
            ]
        },
    ]
    df_formated = (
        df.style
           .set_table_styles(styles)
           .format(formatter="{:.2%}", subset=df.columns[1:])
           .set_properties(**{'min-width': f'{col_width}px'})
        )

    display(df_formated)

set the 'top_n_compute_kernels', it means how many top compute kernels you want to display, default is 8.

In [ ]:
top_n_compute_kernels = 8 

df = pd.read_parquet('comm_compute_final_df.parquet')
comm_size = df[df['type'] == 'comm'].shape[0]
compute_size = df[df['type'] == 'compute'].shape[0]
compute_show_size = min(compute_size, top_n_compute_kernels)

column_list=[0,1] + list(range(comm_size+2, comm_size + 2 + compute_show_size ))

display the communicate compute matrix, if the 'total ** overlap' column large than 100%, it's because some duration is overlaped more than 1 time. 

In [ ]:
comm_compute_df = df.iloc[:comm_size, column_list].set_index('shortName')
comm_compute_df["total_comm_compute_overlap"] = comm_compute_df.iloc[:, 1:].sum(axis=1)
comm_compute_df.index.name = None
formated_display(comm_compute_df)

display the communication communication, if the 'total ** overlap' column large than 100%, it's because some duration is overlaped more than 1 time. 

In [ ]:
comm_comm_df = df.iloc[:comm_size, :comm_size + 2].set_index('shortName')
comm_comm_df["total_comm_comm_overlap"] = comm_comm_df.iloc[:, 1:].sum(axis=1)
comm_comm_df.index.name = None
formated_display(comm_comm_df)

display the compute compute matrix, if the 'total ** overlap' column large than 100%, it's because some duration is overlaped more than 1 time. 

In [ ]:
compute_compute_df = df.iloc[comm_size:comm_size + top_n_compute_kernels , column_list].set_index('shortName')
compute_compute_df["total_compute_compute_overlap"] = compute_compute_df.iloc[:, 1:].sum(axis=1)
compute_compute_df.index.name = None
formated_display(compute_compute_df)

The table associates each rank number with the original filename. Ranks are assigned assuming that the file names include the rank with sufficient zero padding for proper sorting. Otherwise, the actual rank may differ from the assigned ID.

In [ ]:
files_df = pd.read_parquet("files.parquet")
display(files_df)